In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import librosa
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class ICBHIDatasetProcessor:
    """Process ICBHI dataset for GNN training"""
    
    def __init__(self, data_path, target_sr=4000, duration=10):
        self.data_path = Path(data_path)
        self.target_sr = target_sr
        self.duration = duration
        self.target_samples = int(duration * target_sr)
        
        # Load metadata
        self.load_metadata()
        
    def load_metadata(self):
        """Load all metadata files"""
        # Load patient diagnoses
        self.diagnosis_df = pd.read_csv(
            self.data_path / 'patient_diagnosis.txt',
            sep='\t', header=None, names=['patient_id', 'diagnosis']
        )
        
        # Load respiratory cycle annotations
        self.annotations_df = pd.read_csv(
            self.data_path / 'respiratory_cycle_annotation.txt',
            sep='\t', header=None,
            names=['filename', 'start', 'end', 'crackles', 'wheezes']
        )
        
        # Create label encoder for diseases
        self.label_encoder = LabelEncoder()
        self.diagnosis_df['label'] = self.label_encoder.fit_transform(self.diagnosis_df['diagnosis'])
        
        print(f"✅ Loaded {len(self.diagnosis_df)} patients")
        print(f"✅ Disease distribution:")
        print(self.diagnosis_df['diagnosis'].value_counts())
        print(f"✅ Classes: {list(self.label_encoder.classes_)}")
        
    def extract_enhanced_features(self, audio, sr):
        """Extract comprehensive audio features for respiratory analysis"""
        try:
            # MFCC features (core for speech/respiratory analysis)
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13, n_fft=1024, hop_length=512)
            mfcc_delta = librosa.feature.delta(mfcc)
            mfcc_delta2 = librosa.feature.delta(mfcc, order=2)
            
            # Spectral features
            spectral_centroid = librosa.feature.spectral_centroid(y=audio, sr=sr)
            spectral_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr)
            spectral_contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)
            spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)
            
            # Temporal features
            zero_crossing_rate = librosa.feature.zero_crossing_rate(audio)
            rms = librosa.feature.rms(y=audio)
            
            # Tonal features
            chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
            
            # Mel-spectrogram
            mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=13)
            mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            
            # Tempo and rhythm (useful for breathing patterns)
            tempo_features = np.array([[librosa.beat.tempo(y=audio, sr=sr)[0]]] * mfcc.shape[1])
            
            # Stack all features
            features = np.vstack([
                mfcc,                    # 13 features
                mfcc_delta,              # 13 features  
                mfcc_delta2,             # 13 features
                spectral_centroid,       # 1 feature
                spectral_rolloff,        # 1 feature
                spectral_contrast,       # 7 features
                spectral_bandwidth,      # 1 feature
                zero_crossing_rate,      # 1 feature
                chroma,                  # 12 features
                mel_spec,                # 13 features
                rms,                     # 1 feature
                tempo_features.T         # 1 feature
            ])
            
            print(f"Extracted features shape: {features.shape} (should be ~76 x time_frames)")
            return features
            
        except Exception as e:
            print(f"Error extracting features: {e}")
            return None

class FeatureToGraphConverter:
    """Convert audio features to graph representation"""
    
    def __init__(self, k_neighbors=5):
        self.k_neighbors = k_neighbors

    def features_to_graph(self, features, label=None):
        """Convert features matrix to PyTorch Geometric graph"""
        # Transpose to get (time_frames, features)
        features = features.T
        
        # Create k-NN graph
        edge_index = self._create_knn_graph(features)
        
        # Convert to tensors
        x = torch.FloatTensor(features)
        edge_index = torch.LongTensor(edge_index)
        
        # Create graph data object
        if label is not None:
            data = Data(x=x, edge_index=edge_index, y=torch.LongTensor([label]))
        else:
            data = Data(x=x, edge_index=edge_index)
            
        return data

    def _create_knn_graph(self, features):
        """Create k-nearest neighbors graph from features"""
        n_nodes = features.shape[0]
        
        if n_nodes <= 1:
            return np.array([[0], [0]])
            
        # Calculate pairwise distances
        distances = np.linalg.norm(
            features[:, np.newaxis] - features[np.newaxis, :], axis=2
        )
        
        edges = []
        for i in range(n_nodes):
            k_actual = min(self.k_neighbors, n_nodes - 1)
            # Get k nearest neighbors (excluding self)
            neighbors = np.argsort(distances[i])[1:k_actual + 1]
            
            for neighbor in neighbors:
                edges.append([i, neighbor])
                edges.append([neighbor, i])  # Undirected graph
        
        if edges:
            # Remove duplicates and convert to numpy
            edges = list(set([tuple(edge) for edge in edges]))
            edges = np.array(edges).T
        else:
            edges = np.array([[0], [0]])
            
        return edges

class RespiratoryGNN(nn.Module):
    """Graph Neural Network for respiratory sound classification"""
    
    def __init__(self, input_dim=76, hidden_dim=128, num_classes=7, dropout=0.5):
        super().__init__()
        
        # GCN layers
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.conv4 = GCNConv(hidden_dim, hidden_dim // 2)
        
        # Batch normalization
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.bn4 = nn.BatchNorm1d(hidden_dim // 2)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim // 2, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(dropout * 0.7),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(dropout * 0.5),
            nn.Linear(32, num_classes)
        )

    def forward(self, x, edge_index, batch):
        # GCN layers with residual connections
        x1 = F.relu(self.bn1(self.conv1(x, edge_index)))
        x1 = self.dropout(x1)
        
        x2 = F.relu(self.bn2(self.conv2(x1, edge_index)))
        x2 = self.dropout(x2) + x1  # Residual connection
        
        x3 = F.relu(self.bn3(self.conv3(x2, edge_index)))
        x3 = self.dropout(x3) + x2  # Residual connection
        
        x4 = F.relu(self.bn4(self.conv4(x3, edge_index)))
        
        # Global pooling
        x = global_mean_pool(x4, batch)
        
        # Classification
        x = self.classifier(x)
        return F.log_softmax(x, dim=1)

class RespiratoryGNNTrainer:
    """Complete training pipeline for respiratory GNN"""
    
    def __init__(self, dataset_path, test_size=0.2, batch_size=32):
        self.dataset_path = dataset_path
        self.test_size = test_size
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Initialize components
        self.processor = ICBHIDatasetProcessor(dataset_path)
        self.graph_converter = FeatureToGraphConverter(k_neighbors=5)
        
        print(f"🚀 Using device: {self.device}")
        
    def load_and_preprocess_data(self, max_samples_per_class=50):
        """Load audio files and convert to graphs"""
        print("📊 Loading and preprocessing audio data...")
        
        graphs = []
        labels = []
        
        # Get available audio files
        audio_dir = Path(self.dataset_path) / 'audio_and_txt_files'
        available_files = list(audio_dir.glob('*.wav'))
        
        print(f"Found {len(available_files)} audio files")
        
        # Process files with class balancing
        class_counts = {}
        
        for audio_file in tqdm(available_files, desc="Processing audio files"):
            try:
                # Extract patient ID from filename
                patient_id = int(audio_file.stem.split('_')[0])
                
                # Get patient diagnosis
                patient_data = self.processor.diagnosis_df[
                    self.processor.diagnosis_df['patient_id'] == patient_id
                ]
                
                if len(patient_data) == 0:
                    continue
                    
                diagnosis = patient_data.iloc[0]['diagnosis']
                label = patient_data.iloc[0]['label']
                
                # Limit samples per class for balanced dataset
                if class_counts.get(diagnosis, 0) >= max_samples_per_class:
                    continue
                
                # Load and preprocess audio
                audio, sr = librosa.load(
                    audio_file, 
                    sr=self.processor.target_sr,
                    duration=self.processor.duration
                )
                
                if len(audio) == 0:
                    continue
                
                # Normalize audio
                if np.max(np.abs(audio)) > 0:
                    audio = audio / np.max(np.abs(audio))
                
                # Extract features
                features = self.processor.extract_enhanced_features(audio, sr)
                if features is None:
                    continue
                
                # Convert to graph
                graph = self.graph_converter.features_to_graph(features, label)
                
                graphs.append(graph)
                labels.append(label)
                
                # Update class count
                class_counts[diagnosis] = class_counts.get(diagnosis, 0) + 1
                
            except Exception as e:
                print(f"Error processing {audio_file}: {e}")
                continue
        
        print(f"✅ Successfully processed {len(graphs)} audio files")
        print("📊 Class distribution:")
        for diagnosis, count in class_counts.items():
            print(f"  {diagnosis}: {count} samples")
            
        return graphs, labels
    
    def train_model(self, graphs, labels, epochs=100, lr=0.001):
        """Train the GNN model"""
        print("🏋️ Starting model training...")
        
        # Split data
        train_graphs, test_graphs, train_labels, test_labels = train_test_split(
            graphs, labels, test_size=self.test_size, 
            stratify=labels, random_state=42
        )
        
        # Create data loaders
        train_loader = DataLoader(train_graphs, batch_size=self.batch_size, shuffle=True)
        test_loader = DataLoader(test_graphs, batch_size=self.batch_size, shuffle=False)
        
        # Initialize model
        num_classes = len(self.processor.label_encoder.classes_)
        model = RespiratoryGNN(
            input_dim=76,  # Based on feature extraction
            hidden_dim=128,
            num_classes=num_classes,
            dropout=0.5
        ).to(self.device)
        
        # Optimizer and loss
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.NLLLoss()
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
        
        # Training history
        train_losses = []
        train_accuracies = []
        
        best_acc = 0
        best_model_state = None
        
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            correct = 0
            total = 0
            
            for batch in train_loader:
                batch = batch.to(self.device)
                optimizer.zero_grad()
                
                out = model(batch.x, batch.edge_index, batch.batch)
                loss = criterion(out, batch.y)
                
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
                pred = out.argmax(dim=1)
                correct += (pred == batch.y).sum().item()
                total += batch.y.size(0)
            
            scheduler.step()
            
            avg_loss = total_loss / len(train_loader)
            accuracy = correct / total
            
            train_losses.append(avg_loss)
            train_accuracies.append(accuracy)
            
            # Evaluate on test set every 10 epochs
            if (epoch + 1) % 10 == 0:
                test_acc = self.evaluate_model(model, test_loader)
                print(f'Epoch {epoch+1}/{epochs}: '
                      f'Loss: {avg_loss:.4f}, '
                      f'Train Acc: {accuracy:.4f}, '
                      f'Test Acc: {test_acc:.4f}')
                
                # Save best model
                if test_acc > best_acc:
                    best_acc = test_acc
                    best_model_state = model.state_dict().copy()
        
        # Load best model
        model.load_state_dict(best_model_state)
        
        # Final evaluation
        final_test_acc = self.evaluate_model(model, test_loader)
        print(f"\n🎯 Final Test Accuracy: {final_test_acc:.4f}")
        
        # Detailed evaluation
        self.detailed_evaluation(model, test_loader)
        
        return model, train_losses, train_accuracies
    
    def evaluate_model(self, model, test_loader):
        """Evaluate model performance"""
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(self.device)
                out = model(batch.x, batch.edge_index, batch.batch)
                pred = out.argmax(dim=1)
                correct += (pred == batch.y).sum().item()
                total += batch.y.size(0)
        
        return correct / total
    
    def detailed_evaluation(self, model, test_loader):
        """Detailed evaluation with metrics and confusion matrix"""
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(self.device)
                out = model(batch.x, batch.edge_index, batch.batch)
                pred = out.argmax(dim=1)
                
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(batch.y.cpu().numpy())
        
        # Classification report
        print("\n📊 Classification Report:")
        print(classification_report(
            all_labels, all_preds,
            target_names=self.processor.label_encoder.classes_,
            digits=4
        ))
        
        # Confusion matrix
        cm = confusion_matrix(all_labels, all_preds)
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', 
                   xticklabels=self.processor.label_encoder.classes_,
                   yticklabels=self.processor.label_encoder.classes_,
                   cmap='Blues')
        plt.title('Confusion Matrix - Respiratory Disease Classification')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.tight_layout()
        plt.show()

def main():
    """Main training pipeline"""
    # Set path to your ICBHI dataset
    dataset_path = "ICBHI_final_database (1)\ICBHI_final_database"  # Update this path
    
    print("🏥 GNN-based Respiratory Disease Classification")
    print("=" * 50)
    
    # Initialize trainer
    trainer = RespiratoryGNNTrainer(dataset_path, test_size=0.2, batch_size=16)
    
    # Load and preprocess data
    graphs, labels = trainer.load_and_preprocess_data(max_samples_per_class=30)
    
    if len(graphs) == 0:
        print("❌ No data loaded. Check your dataset path.")
        return
    
    # Train model
    model, train_losses, train_accuracies = trainer.train_model(
        graphs, labels, epochs=50, lr=0.001
    )
    
    print("✅ Training completed successfully!")
    
    # Plot training history
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies)
    plt.title('Training Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

🏥 GNN-based Respiratory Disease Classification


FileNotFoundError: [Errno 2] No such file or directory: 'ICBHI_final_database (1)\\ICBHI_final_database\\patient_diagnosis.txt'